In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [10]:
# Install Java
!apt-get update
!apt-get install -y openjdk-11-jdk-headless
!java -version

# Download Stanford CoreNLP 4.5.5
!wget https://nlp.stanford.edu/software/stanford-corenlp-4.5.5.zip
!unzip -o stanford-corenlp-4.5.5.zip

Hit:1 https://packages.cloud.google.com/apt gcsfuse-focal InRelease
Hit:2 http://archive.ubuntu.com/ubuntu focal InRelease
Hit:3 http://security.ubuntu.com/ubuntu focal-security InRelease
Hit:4 https://packages.cloud.google.com/apt cloud-sdk InRelease
Hit:5 http://archive.ubuntu.com/ubuntu focal-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu focal-backports InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree       
Reading state information... Done
openjdk-11-jdk-headless is already the newest version (11.0.24+8-1ubuntu3~20.04).
0 upgraded, 0 newly installed, 0 to remove and 58 not upgraded.
openjdk version "11.0.24" 2024-07-16
OpenJDK Runtime Environment (build 11.0.24+8-post-Ubuntu-1ubuntu320.04)
OpenJDK 64-Bit Server VM (build 11.0.24+8-post-Ubuntu-1ubuntu320.04, mixed mode, sharing)
--2024-11-04 03:48:16--  https://nlp.stanford.edu/software/stanford-corenlp-4.5.5.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140


In [11]:
!wget http://nlp.stanford.edu/software/stanford-corenlp-4.5.5-models-english.jar

--2024-11-04 03:49:53--  http://nlp.stanford.edu/software/stanford-corenlp-4.5.5-models-english.jar
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/software/stanford-corenlp-4.5.5-models-english.jar [following]
--2024-11-04 03:49:53--  https://nlp.stanford.edu/software/stanford-corenlp-4.5.5-models-english.jar
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 302 FOUND
Location: https://downloads.cs.stanford.edu/nlp/software/stanford-corenlp-4.5.5-models-english.jar [following]
--2024-11-04 03:49:53--  https://downloads.cs.stanford.edu/nlp/software/stanford-corenlp-4.5.5-models-english.jar
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|

In [12]:
!export CLASSPATH=$CLASSPATH:/kaggle/working/stanford-corenlp-4.5.5/*:/kaggle/working/stanford-corenlp-4.5.5-models-english.jar

In [13]:
import subprocess

try:
    result = subprocess.run(["jar", "tf", "/kaggle/working/stanford-corenlp-4.5.5/stanford-corenlp-4.5.5.jar"], 
                            capture_output=True, text=True, check=True)
    merge_nodes_entries = [line for line in result.stdout.split('\n') if 'MergeNodes' in line]
    
    if merge_nodes_entries:
        print("MergeNodes entries found:")
        for entry in merge_nodes_entries:
            print(entry)
    else:
        print("No MergeNodes entries found in the JAR file.")
except subprocess.CalledProcessError as e:
    print(f"An error occurred while executing the jar command: {e}")
except FileNotFoundError:
    print("The jar command was not found. Make sure Java is installed and in your PATH.")

MergeNodes entries found:
edu/stanford/nlp/semgraph/semgrex/ssurgeon/MergeNodes.class


In [23]:

import subprocess
import os

java_code = '''
import edu.stanford.nlp.pipeline.*;
import edu.stanford.nlp.semgraph.*;
import edu.stanford.nlp.semgraph.semgrex.*;
import edu.stanford.nlp.ling.*;
import edu.stanford.nlp.util.*;
import java.util.*;

public class FashionSemanticSearch {
    private final StanfordCoreNLP pipeline;
    private final List<String> semgrexPatterns;
    
    public FashionSemanticSearch() {
        Properties props = new Properties();
        props.setProperty("annotators", "tokenize,ssplit,pos,lemma,parse,depparse");
        this.pipeline = new StanfordCoreNLP(props);
        this.semgrexPatterns = initializePatterns();
    }
    
    private List<String> initializePatterns() {
        List<String> patterns = new ArrayList<>();
        patterns.add("{word:/wear|dress/} >obl:to ({word:/party|wedding|dinner|meeting|interview/}=occasion)");
        patterns.add("{} >obl:to ({word:/party|meeting/}=occasion >compound {}=occasion_type)");
        patterns.add("{} >obl:to ({word:/party|dinner|meeting/}=occasion >amod {}=style)");
        patterns.add("{} >obl:to ({word:/wedding|party/}=occasion >compound {word:/summer|winter|spring|fall|autumn/}=season)");
        patterns.add("{} >obl:to ({word:/party|dinner/}=occasion >compound {word:/evening|morning|afternoon|night/}=time)");
        return patterns;
    }
    
    public FashionSearchResult analyzeFashionQuery(String query) {
        Annotation document = new Annotation(query);
        pipeline.annotate(document);
        FashionSearchResult result = new FashionSearchResult();
        
        for (CoreMap sentence : document.get(CoreAnnotations.SentencesAnnotation.class)) {
            SemanticGraph graph = sentence.get(SemanticGraphCoreAnnotations.EnhancedPlusPlusDependenciesAnnotation.class);
            System.out.println("\\nDependency Graph:");
            System.out.println(graph.toString());
            
            for (String patternStr : semgrexPatterns) {
                try {
                    SemgrexPattern pattern = SemgrexPattern.compile(patternStr);
                    SemgrexMatcher matcher = pattern.matcher(graph);
                    while (matcher.find()) {
                        processMatch(matcher, result);
                    }
                } catch (Exception e) {
                    System.err.println("Error with pattern: " + patternStr);
                    e.printStackTrace();
                }
            }
        }
        return result;
    }
    
    private void processMatch(SemgrexMatcher matcher, FashionSearchResult result) {
        for (String nodeName : matcher.getNodeNames()) {
            IndexedWord node = matcher.getNode(nodeName);
            String word = node.word().toLowerCase();
            switch (nodeName) {
                case "occasion": result.setOccasion(word); break;
                case "occasion_type": result.addOccasionModifier(word); break;
                case "style": result.addStylePreference(word); break;
                case "season": result.setWeatherContext(word); break;
                case "time": result.setTimeContext(word); break;
            }
        }
    }
    
    public static void main(String[] args) {
        FashionSemanticSearch searcher = new FashionSemanticSearch();
        String[] queries = {
            "What should I wear to a bachelor party?",
            "I need something to wear to a summer wedding",
            "What's appropriate for a formal business meeting?",
            "What should I wear to an evening dinner party?",
            "Looking for an outfit for a casual weekend brunch"
        };
        
        for (String query : queries) {
            System.out.println("\\n=== Analyzing query: " + query + " ===");
            FashionSearchResult result = searcher.analyzeFashionQuery(query);
            System.out.println(result.toString());
        }
    }
}

class FashionSearchResult {
    private String occasion;
    private final Set<String> occasionModifiers = new HashSet<>();
    private final Set<String> stylePreferences = new HashSet<>();
    private String weatherContext;
    private String timeContext;
    
    public void setOccasion(String occasion) { this.occasion = occasion; }
    public void addOccasionModifier(String modifier) { this.occasionModifiers.add(modifier); }
    public void addStylePreference(String style) { this.stylePreferences.add(style); }
    public void setWeatherContext(String weather) { this.weatherContext = weather; }
    public void setTimeContext(String time) { this.timeContext = time; }
    
    @Override
    public String toString() {
        StringBuilder sb = new StringBuilder();
        sb.append("Analysis Results:\\n");
        sb.append("- Base Occasion: ").append(occasion != null ? occasion : "not specified").append("\\n");
        if (!occasionModifiers.isEmpty()) {
            sb.append("- Occasion Modifiers: ").append(String.join(", ", occasionModifiers)).append("\\n");
        }
        if (!stylePreferences.isEmpty()) {
            sb.append("- Style Preferences: ").append(String.join(", ", stylePreferences)).append("\\n");
        }
        if (weatherContext != null) {
            sb.append("- Weather/Season: ").append(weatherContext).append("\\n");
        }
        if (timeContext != null) {
            sb.append("- Time Context: ").append(timeContext).append("\\n");
        }
        sb.append("\\nRecommended Outfit:\\n");
        for (String item : generateRecommendations()) {
            sb.append("- ").append(item).append("\\n");
        }
        return sb.toString();
    }
    
    private List<String> generateRecommendations() {
        List<String> recommendations = new ArrayList<>();
        if (occasion != null) {
            switch (occasion) {
                case "party":
                    if (occasionModifiers.contains("bachelor")) {
                        recommendations.addAll(Arrays.asList(
                            "Button-down shirt",
                            "Dark jeans or chinos",
                            "Leather dress shoes",
                            "Sport watch"
                        ));
                    } else if (timeContext != null && timeContext.equals("evening")) {
                        recommendations.addAll(Arrays.asList(
                            "Dark blazer",
                            "Dress shirt",
                            "Tailored pants",
                            "Oxford shoes"
                        ));
                    }
                    break;
                case "wedding":
                    if (weatherContext != null && weatherContext.equals("summer")) {
                        recommendations.addAll(Arrays.asList(
                            "Light colored suit",
                            "White dress shirt",
                            "Pastel tie",
                            "Brown leather shoes"
                        ));
                    }
                    break;
                case "meeting":
                    if (occasionModifiers.contains("business")) {
                        recommendations.addAll(Arrays.asList(
                            "Business suit",
                            "Crisp white shirt",
                            "Conservative tie",
                            "Black Oxford shoes"
                        ));
                    }
                    break;
            }
        }
        if (recommendations.isEmpty()) {
            recommendations.add("No specific recommendations available for this combination");
        }
        return recommendations;
    }
}
'''

# Write the Java code to a file
with open('FashionSemanticSearch.java', 'w') as f:
    f.write(java_code)

print("Created Java file")

# Set the classpath to include the current directory and Stanford CoreNLP JARs
classpath = ".:/kaggle/working/stanford-corenlp-4.5.5/*"

# Compile the Java code
print("Compiling Java code...")
compile_command = ["javac", "-encoding", "UTF-8", "-cp", classpath, "FashionSemanticSearch.java"]
compile_result = subprocess.run(compile_command, capture_output=True, text=True)

if compile_result.returncode == 0:
    print("Compilation successful")
    
    # Run the Java program
    print("Running program...")
    run_command = ["java", "-cp", classpath, "FashionSemanticSearch"]
    run_result = subprocess.run(run_command, capture_output=True, text=True)
    
    print("\nProgram output:")
    print(run_result.stdout)
    
    if run_result.stderr:
        print("\nErrors or warnings:")
        print(run_result.stderr)
else:
    print("Compilation failed:")
    print(compile_result.stderr)


Created Java file
Compiling Java code...
Compilation successful
Running program...

Program output:

=== Analyzing query: What should I wear to a bachelor party? ===

Dependency Graph:
-> wear/VB (root)
  -> What/WP (dep)
  -> should/MD (aux)
  -> I/PRP (nsubj)
  -> party/NN (obl:to)
    -> to/IN (case)
    -> a/DT (det)
    -> bachelor/NN (compound)
  -> ?/. (punct)

Analysis Results:
- Base Occasion: not specified

Recommended Outfit:
- No specific recommendations available for this combination


=== Analyzing query: I need something to wear to a summer wedding ===

Dependency Graph:
-> need/VBP (root)
  -> I/PRP (nsubj)
  -> something/NN (obj)
  -> wear/VB (xcomp)
    -> something/NN (nsubj:xsubj)
    -> to/TO (mark)
    -> wedding/NN (obl:to)
      -> to/IN (case)
      -> a/DT (det)
      -> summer/NN (compound)

Analysis Results:
- Base Occasion: not specified

Recommended Outfit:
- No specific recommendations available for this combination


=== Analyzing query: What's appropria